# 03 Product Category Analysis

## Project Objectives

- Identify high-performing and low-performing product categories
- Analyse category sales, satisfaction, and customer preference
- Support category operation and inventory decisions with data evidence

## 1. Data Loading & Initial Exploration

In [1]:
# Import library code units
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.unicode_minus"] = False

### 1.1 Category-level Data Loading

To analyse product category performance, the first step is to load a category-level aggregated dataset from the MySQL data warehouse. This table summarises the key business indicators for each product category, including sales volume, customer count, revenue, freight cost, review score, bad review rate, repurchase proxy, and category lifecycle. Since the data is already aggregated at the category level, each row represents one product category rather than one individual order or one customer.

In [3]:
# load category-level aggregated data from MySQL
# each row in this dataset represents one product category
from src.utils.db import get_engine

engine = get_engine()

sql_category = """
SELECT *
FROM view_category_analysis
"""

df_category = pd.read_sql(sql_category, engine)

print(df_category.shape)
df_category.head()

(73, 14)


,category,order_count,customer_count,total_revenue,total_gmv,avg_price,avg_freight,avg_review_score,bad_review_rate,avg_comment_len,repeat_rate,first_sale_date,last_sale_date,category_lifetime_days
0,agro_industria_e_comercio,182,182,72530.47,78374.07,342.124858,27.564151,4.0000,16.03774,20.7028,1.0,2017-01-23 07:03:04,2018-08-26 07:57:32,580
1,alimentos,450,450,29393.41,36664.44,57.634137,14.256922,4.2134,13.00813,22.6199,1.0,2016-10-10 11:22:36,2018-08-29 11:06:11,688
2,alimentos_bebidas,227,227,15218.47,19753.64,54.546487,16.255090,4.3091,8.00000,20.7891,1.0,2017-03-05 01:03:51,2018-08-23 19:57:47,536
3,artes,202,202,24202.64,28247.81,115.802105,19.354880,3.9515,17.96117,31.5777,1.0,2017-03-01 10:56:53,2018-08-27 19:37:48,544
4,artes_e_artesanato,23,23,1814.01,2184.14,75.583750,15.422083,4.1250,12.50000,43.4167,1.0,2017-05-08 12:01:54,2018-08-24 12:41:33,473


In [4]:
# View the fields and data types of the df_category table.
df_category.info()

<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   category                73 non-null     str           
 1   order_count             73 non-null     int64         
 2   customer_count          73 non-null     int64         
 3   total_revenue           73 non-null     float64       
 4   total_gmv               73 non-null     float64       
 5   avg_price               73 non-null     float64       
 6   avg_freight             73 non-null     float64       
 7   avg_review_score        73 non-null     float64       
 8   bad_review_rate         73 non-null     float64       
 9   avg_comment_len         73 non-null     float64       
 10  repeat_rate             73 non-null     float64       
 11  first_sale_date         73 non-null     datetime64[us]
 12  last_sale_date          73 non-null     datetime64[us]
 13  cat

In [5]:
df_category.columns.tolist()

['category',
 'order_count',
 'customer_count',
 'total_revenue',
 'total_gmv',
 'avg_price',
 'avg_freight',
 'avg_review_score',
 'bad_review_rate',
 'avg_comment_len',
 'repeat_rate',
 'first_sale_date',
 'last_sale_date',
 'category_lifetime_days']

In [6]:
df_category["category"].nunique()

73

The category-level dataset contains aggregated information for 73 product categories and 14 variables. This confirms that the table is structured at the category level rather than the transaction level. Such a dataset is suitable for high-level comparative analysis across categories, such as ranking categories by revenue, identifying high- and low-performing groups, and constructing category strategy frameworks such as Pareto analysis and quadrant classification.

### 1.2 Order-level Data Loading
In addition to the category-level aggregated table, this project also requires a more detailed order-level dataset. The category-level table is useful for overall comparison, but it does not preserve transaction-level variation. To analyse pricing patterns, customer preference, delivery-related variables, satisfaction behaviour, and temporal trends, an order-item level dataset is loaded by joining product, order, review, and customer information from the warehouse.

In [8]:
# load order_item level data for detailed category analysis
# each row in this dataset represents an order item linked to a category
sql_orders = """
SELECT
    p.category,
    oi.order_id,
    o.user_id,
    oi.product_id,
    oi.price,
    oi.freight_value,
    oi.gmv,
    o.purchase_ts,
    o.delivered_days,
    r.review_score,
    r.review_comment_len,
    u.state AS customer_state
FROM fact_order_item oi
JOIN dim_product p
    ON oi.product_id = p.product_id
JOIN fact_order o
    ON oi.order_id = o.order_id
LEFT JOIN fact_review r
    ON o.order_id = r.order_id
LEFT JOIN dim_user u
    ON o.user_id = u.user_id
WHERE p.category IS NOT NULL
"""

# read sql result into pandas dataframe
df_orders = pd.read_sql(sql_orders, engine)

# check dataset shape
print(df_orders.shape)

# preview the first few rows
df_orders.head()

(111426, 12)


,category,order_id,user_id,product_id,price,freight_value,gmv,purchase_ts,delivered_days,review_score,review_comment_len,customer_state
0,cool_stuff,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,4244733e06e7ecb4970a6e2683c13e61,58.90,13.29,72.19,2017-09-13 08:59:02,7.0,5.0,46.0,RJ
1,pet_shop,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,e5f2d52b802189ee658865ca93d83a8f,239.90,19.93,259.83,2017-04-26 10:53:06,16.0,4.0,0.0,SP
2,moveis_decoracao,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,c777355d18b72b67abbeef9df44fd0fd,199.00,17.87,216.87,2018-01-14 14:33:31,8.0,5.0,90.0,MG
3,perfumaria,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,7634da152a4610f1595efa32f14722fc,12.99,12.79,25.78,2018-08-08 10:00:35,6.0,4.0,0.0,SP
4,ferramentas_jardim,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,ac6c3623068f30de03045865e4e10089,199.90,18.14,218.04,2017-02-04 13:57:51,25.0,5.0,39.0,SP


The order-level dataset preserves detailed transaction records and is much larger than the aggregated category-level dataset. Each row corresponds to an order item associated with a product category, customer, and review context. This dataset is necessary for downstream analyses that cannot be carried out at the grouped category level, including price distribution analysis, category preference by user segment, monthly sales trend analysis, state-level category comparison, and selected statistical tests.

### 1.3 Data Structure Check
Before starting the formal analysis, it is necessary to check the structure and data quality of the two datasets. This includes examining variable types, identifying data fields, and check whether important variables contain missing values. These checks help ensure that the following analysis is based on correctly formatted and reliable data.

In [10]:
# check the structure, data types, and non-null counts of both datasets

print("Category-level dataset info:")
df_category.info()

print("\n" + "=" * 60 + "\n")

print("Order-level dataset info:")
df_orders.info()

Category-level dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 73 entries, 0 to 72
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   category                73 non-null     str           
 1   order_count             73 non-null     int64         
 2   customer_count          73 non-null     int64         
 3   total_revenue           73 non-null     float64       
 4   total_gmv               73 non-null     float64       
 5   avg_price               73 non-null     float64       
 6   avg_freight             73 non-null     float64       
 7   avg_review_score        73 non-null     float64       
 8   bad_review_rate         73 non-null     float64       
 9   avg_comment_len         73 non-null     float64       
 10  repeat_rate             73 non-null     float64       
 11  first_sale_date         73 non-null     datetime64[us]
 12  last_sale_date          73 non-nul

In [11]:
# check missing values in both datasets

print("Missing values in df_category:")
print(df_category.isnull().sum())

print("\n" + "=" * 60 + "\n")

print("Missing values in df_orders:")
print(df_orders.isnull().sum())

Missing values in df_category:
category                  0
order_count               0
customer_count            0
total_revenue             0
total_gmv                 0
avg_price                 0
avg_freight               0
avg_review_score          0
bad_review_rate           0
avg_comment_len           0
repeat_rate               0
first_sale_date           0
last_sale_date            0
category_lifetime_days    0
dtype: int64


Missing values in df_orders:
category                 0
order_id                 0
user_id                  0
product_id               0
price                    0
freight_value            0
gmv                      0
purchase_ts              0
delivered_days        2399
review_score          1572
review_comment_len    1572
customer_state           0
dtype: int64


### Data Structure Check analysis and summary
The data structure check shows that the category-level dataset is clean, compact, and already well aggregated. It contains 73 product categories and 14 variables, including sales indicators, customer-related metrics, review-related measures, and lifecycle information. No missing values are found in this dataset, and both `first_sale_date` and `last_sale_date` have already been correctly recognised as datetime variables. This means the category-level table is ready for direct use in category comparison, ranking analysis, Pareto analysis, and quadrant-based strategic classification.

The order-level dataset contains 111,426 transaction records and 12 variables, which provide a much more detailed view of category behaviour. Most key business fields, including category, order ID, user ID, product ID, price, freight value, GMV, purchase timestamp, and customer state, are complete. The `purchase_ts` field has also been correctly stored as a datetime variable, which is suitable for later time-series analysis.

However, some missing values are present in the order-level dataset. Specifically, `delivered_days` contains 2,399 missing values, while `review_score` and `review_comment_len` each contain 1,572 missing values. These missing values are expected in an e-commerce context. Missing delivery information usually corresponds to orders that were not fully delivered or whose delivery status was incomplete at the time of data recording. Missing review-related values are also common because not all customers leave a review after purchase. Therefore, these missing values should not be removed blindly from the full dataset. Instead, they should be handled selectively depending on the analytical objective of each later section.

### 1.4 Data Preparation
Before moving to exploratory analysis, a small amount of data preparation is carried out. Since the full order-level dataset will support multiple downstream tasks, it is useful to prepare separate subsets for review-related analysis and delivery-related analysis.This avoids unnecessary global data deletion and keeps the original transaction dataset intact for later use.

In [13]:
# create analysis-specific subsets from the full order-level dataset
# keep the original df_orders unchanged

# subset for review-related analysis
df_review = df_orders[df_orders["review_score"].notna()].copy()

# subset for delivery-related analysis
df_delivery = df_orders[df_orders["delivered_days"].notna()].copy()

# subset for analysis requiring both review and delivery information
df_review_delivery = df_orders[
    df_orders["review_score"].notna() &
    df_orders["delivered_days"].notna()
].copy()

# check the size of each subset
print("Full order-level dataset:", df_orders.shape)
print("Review subset:", df_review.shape)
print("Delivery subset:", df_delivery.shape)
print("Review + Delivery subset:", df_review_delivery.shape)

Full order-level dataset: (111426, 12)
Review subset: (109854, 12)
Delivery subset: (109027, 12)
Review + Delivery subset: (107587, 12)


Instead of removing missing values from the full order-level dataset, task-specific subsets were created for later analysis. The full order-level dataset contains 111,426 records. Among them, 109,854 records have valid review information, 109,027 records have valid delivery duration information, and 107,587 records contain both review and delivery information. This result shows that the proportion of missing values is relatively small compared with the full dataset. Therefore, using separate subsets is a suitable strategy because it preserves the full transaction table for general category analysis while allowing more focused analysis when review or delivery variables are required.

## 2. Exploratory Data Analysis
Exploratory data analysis is carried out to understand the structure, distribution, and reliability of the category-level and order-level datasets before moving to business interpretation. This stage focuses on missing values, variable distributions, which help identify potential data issues and provide cntext for later comparative analysis.

### 2.1 Missing Value Summary and Interpretation
The first step in exploratory analysis is to review missing values in both datasets. Missing-value patterns are important because they affect which records can be used in later analysis, especially for review-related and delivery-related variables.

In [15]:
# summarise missing values in both datasets

missing_category = pd.DataFrame({
    "missing_count": df_category.isnull().sum(),
    "missing_pct": df_category.isnull().sum() * 100
})

missing_orders = pd.DataFrame({
    "missing_count": df_orders.isnull().sum(),
    "missing_pct": df_orders.isnull().mean() * 100
})

print("Missing values in category-level dataset:")
display(missing_category[missing_category["missing_count"] > 0])

print("\nMissing values in order-level dataset:")
display(missing_orders[missing_orders["missing_count"] > 0].sort_values("missing_count", ascending=False))

Missing values in category-level dataset:


,missing_count,missing_pct



Missing values in order-level dataset:


,missing_count,missing_pct
delivered_days,2399,2.152998
review_score,1572,1.410802
review_comment_len,1572,1.410802
